# A Good Fit Is Not an Identified Model

Executable companion to the long-form [inverse problems article](article.md). This notebook uses an entirely **synthetic** diffusion-like smoothing operator and a smaller fixed-seed Monte Carlo experiment. The physical signal is known only because the code generates it. Run `python analysis.py` for all eight full publication figures and ten CSV result tables.

## 1. Structural non-identifiability versus numerical ill-conditioning

Observing 20 linear measurements of 64 latent values leaves a nontrivial exact nullspace. Observing all 64 values through the smoothing operator gives a full-rank matrix with extremely small high-frequency singular values. These are related but distinct problems.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from analysis import N, SENSOR_BUDGET, run_study

study = run_study(quick=True)
delta = study.alternate_truth - study.truth
print('Latent L2 distance:', round(float(np.linalg.norm(delta)), 6))
print('Observed sensor difference:', np.linalg.norm(study.operator[study.sparse_indices] @ delta))
assert len(study.sparse_indices) == SENSOR_BUDGET
assert np.linalg.norm(study.operator[study.sparse_indices] @ delta) < 1e-10

## 2. Ill-conditioned inverses and spectral regularization

The direct inverse fits the noisy observed vector almost exactly, but the latent signal is badly reconstructed. Truncated SVD and Tikhonov regularization suppress unstable directions. GCV selects a regularization parameter without access to the latent truth; the oracle error is available for diagnostic comparison **only in this simulation**.

In [ ]:
print('Condition number:', round(study.summary['condition_number']))
for label in ('ls_latent_rmse', 'tsvd_latent_rmse', 'tikhonov_latent_rmse', 'oracle_latent_rmse'):
    print(label, round(study.summary[label], 5))
fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(study.grid, study.truth, 'k-', label='Known truth')
ax.plot(study.grid, study.tsvd_solution, label='TSVD')
ax.plot(study.grid, study.tikhonov_solution, label='GCV Tikhonov')
ax.set_xlabel('Position')
ax.set_ylabel('Latent signal')
ax.legend()
ax.grid(alpha=0.2)
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3.5))
ax.semilogx(study.alpha_grid, study.truth_errors, label='Oracle latent RMSE')
ax.axvline(study.alpha_gcv, ls='--', label='GCV choice')
ax.axvline(study.alpha_oracle, ls=':', label='Oracle choice')
ax.set_xlabel('Regularization strength')
ax.set_ylabel('True error, available only in simulation')
ax.grid(alpha=0.2)
ax.legend()
plt.show()

## 3. Bayesian uncertainty is conditional on its prior

The Gaussian posterior distribution is exact relative to the stated prior, forward operator, and noise model. The fixed synthetic truth is not drawn from this prior. Repeated-noise *frequentist* pointwise coverage therefore need not equal the nominal Bayesian 95% level.

In [ ]:
print('Mean fixed-truth pointwise coverage:', round(float(np.mean(study.pointwise_coverage)), 3))
fig, ax = plt.subplots(figsize=(8, 3.5))
ax.plot(study.grid, study.pointwise_coverage, label='Empirical fixed-truth coverage')
ax.axhline(0.95, ls='--', color='tab:red', label='Nominal Bayesian level')
ax.set_ylim(0, 1.03)
ax.set_xlabel('Position')
ax.set_ylabel('Coverage')
ax.legend()
ax.grid(alpha=0.2)
plt.show()

## 4. Inverse problems begin at sensor selection

All three strategies use exactly twenty measurement locations. The greedy A-optimal strategy is a myopic trace-reduction heuristic; it is not proven globally optimal. Our synthetic prior and physical operator strongly favour spatially distributed observations.

In [ ]:
for name, trace in study.design_trace.items():
    latent, forward = study.sensor_trial_errors[name]
    print(name, 'trace:', round(float(trace[-1]), 4), 'latent RMSE:', round(float(np.mean(latent)), 4), 'forward RMSE:', round(float(np.mean(forward)), 4))
assert all(len(indices) == SENSOR_BUDGET for indices in study.sensor_designs.values())
assert study.summary['ls_latent_rmse'] > study.summary['tikhonov_latent_rmse']

## Scope

This is a finite-dimensional, linear, Gaussian, correctly specified forward map with known noise SD. It does not establish that the same sensor placements, prior, regularization strength or coverage hold for real equipment. See [the article](article.md) and its references for model criticism, uncertainty, selection and experimental-design limitations.